# GFPGAN local para Anaconda + Jupyter (Windows/Linux)

Notebook actualizado para ejecutar **GFPGAN en local** con rutas robustas, descarga de pesos, verificación de GPU y visualización antes/después.

**Recomendado para tu caso:**
- Python 3.10
- PyTorch con CUDA
- Jupyter Notebook o JupyterLab
- GPU NVIDIA GTX 1650 Ti o similar

> Este flujo sigue la instalación e inferencia oficial de GFPGAN: clonar el repositorio, instalar `basicsr`, `facexlib`, `realesrgan` y ejecutar `inference_gfpgan.py`. La documentación oficial indica Python >= 3.7, PyTorch >= 1.7, e inferencia con modelos v1, v1.2, v1.3 y soporte añadido para v1.4/RestoreFormer. citeturn391527view0turn859644view0turn799919search5

## 1. Diagnóstico del entorno

Esta celda:
- muestra versión de Python
- confirma qué ejecutable usa Jupyter
- revisa PyTorch
- confirma si CUDA está activa
- imprime la GPU detectada

In [1]:
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Ejecutable:", sys.executable)
print("Sistema operativo:", platform.platform())

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA runtime:", torch.version.cuda)
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("GPU: no disponible en este entorno")
except Exception as e:
    print("Torch no está disponible:", repr(e))

Python: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
Ejecutable: C:\Users\VICTORIA\anaconda3\envs\imagen\python.exe
Sistema operativo: Windows-10-10.0.22631-SP0
Torch: 2.7.1+cu118
CUDA runtime: 11.8
CUDA disponible: True
GPU: GeForce GTX 1650 Ti


## 2. Notas importantes sobre GPU

Si aquí aparece algo como `Torch: ...+cpu` o `CUDA disponible: False`, este notebook puede correr, pero **sin aceleración GPU**.

Para una instalación con GPU en Windows con pip, PyTorch publica comandos oficiales con ruedas CUDA 11.8, 12.6 y 12.8. Para una GTX 1650 Ti, una opción conservadora es `cu118`. citeturn799919search2turn799919search5

Ejemplo:

```bash
pip uninstall torch torchvision torchaudio -y
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
```

Después reinicia el kernel y vuelve a ejecutar la celda 1.

## 3. Configuración de rutas locales

Puedes cambiar `PROJECT_ROOT` si quieres guardar el proyecto en otra carpeta.

In [3]:
from pathlib import Path
import os

# Cambia esta ruta si quieres otro directorio local
PROJECT_ROOT = Path.home() 
REPO_DIR = PROJECT_ROOT / "GFPGAN"
INPUT_DIR = REPO_DIR / "inputs" / "user_images"
OUTPUT_DIR = REPO_DIR / "results_local"
WEIGHTS_DIR_1 = REPO_DIR / "experiments" / "pretrained_models"
WEIGHTS_DIR_2 = REPO_DIR / "gfpgan" / "weights"

for p in [PROJECT_ROOT, INPUT_DIR, OUTPUT_DIR, WEIGHTS_DIR_1, WEIGHTS_DIR_2]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("REPO_DIR     =", REPO_DIR)
print("INPUT_DIR    =", INPUT_DIR)
print("OUTPUT_DIR   =", OUTPUT_DIR)
print("WEIGHTS_DIR_1=", WEIGHTS_DIR_1)
print("WEIGHTS_DIR_2=", WEIGHTS_DIR_2)

PROJECT_ROOT = C:\Users\VICTORIA
REPO_DIR     = C:\Users\VICTORIA\GFPGAN
INPUT_DIR    = C:\Users\VICTORIA\GFPGAN\inputs\user_images
OUTPUT_DIR   = C:\Users\VICTORIA\GFPGAN\results_local
WEIGHTS_DIR_1= C:\Users\VICTORIA\GFPGAN\experiments\pretrained_models
WEIGHTS_DIR_2= C:\Users\VICTORIA\GFPGAN\gfpgan\weights


## 4. Utilidades de ejecución

Se define una función para ejecutar comandos del sistema y mostrar salida completa.

In [5]:
import subprocess
import shlex

def run_command(cmd, cwd=None, check=True):
    """
    Ejecuta un comando y transmite la salida en vivo.
    cmd puede ser lista o string.
    """
    if isinstance(cmd, str):
        shell = True
        printable = cmd
    else:
        shell = False
        printable = " ".join(map(str, cmd))
    print(f"\n[Ejecutando] {printable}\n")
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=shell,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        universal_newlines=True
    )
    lines = []
    for line in process.stdout:
        print(line, end="")
        lines.append(line)
    ret = process.wait()
    if check and ret != 0:
        raise RuntimeError(f"El comando falló con código {ret}\n{''.join(lines[-50:])}")
    return ret, "".join(lines)

## 5. Clonar o actualizar el repositorio oficial

La guía oficial indica:
1. `git clone`
2. instalar dependencias
3. `python setup.py develop`
4. ejecutar `inference_gfpgan.py` sobre una carpeta o imagen local. citeturn391527view0

In [6]:
from pathlib import Path

REPO_DIR = Path.home() / "GFPGAN_repo"

if not REPO_DIR.exists():
    run_command(["git", "clone", "https://github.com/TencentARC/GFPGAN.git", str(REPO_DIR)])
    print(f"Repositorio clonado en: {REPO_DIR}")

elif (REPO_DIR / ".git").exists():
    print(f"Repositorio ya existe en: {REPO_DIR}")
    try:
        run_command(["git", "pull"], cwd=REPO_DIR, check=False)
    except Exception as e:
        print("Aviso: no se pudo actualizar con git pull:", e)

else:
    raise RuntimeError(
        f"La carpeta {REPO_DIR} ya existe pero no es un repositorio Git válido.\n"
        "Renómbrala, elimínala o cambia REPO_DIR a otra ruta antes de continuar."
    )


[Ejecutando] git clone https://github.com/TencentARC/GFPGAN.git C:\Users\VICTORIA\GFPGAN_repo

Cloning into 'C:\Users\VICTORIA\GFPGAN_repo'...
Repositorio clonado en: C:\Users\VICTORIA\GFPGAN_repo


## 6. Instalación de dependencias del notebook y del proyecto

Esta celda:
- actualiza `pip`
- instala dependencias auxiliares del notebook
- instala `basicsr`, `facexlib`
- instala los requisitos del repositorio
- instala `realesrgan` de forma opcional
- registra el repositorio en modo editable

La documentación oficial recomienda instalar `basicsr`, `facexlib`, `requirements.txt`, `python setup.py develop`, y `realesrgan` si quieres mejorar fondo. citeturn391527view0

In [7]:
# Dependencias mínimas del notebook
run_command([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
run_command([sys.executable, "-m", "pip", "install", "matplotlib", "opencv-python", "pillow", "numpy", "ipywidgets", "requests"])

# Dependencias sugeridas por el repositorio oficial
run_command([sys.executable, "-m", "pip", "install", "basicsr", "facexlib"], cwd=REPO_DIR)
run_command([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO_DIR)
run_command([sys.executable, "-m", "pip", "install", "realesrgan"], cwd=REPO_DIR)

# Instalación editable del paquete local
run_command([sys.executable, "setup.py", "develop"], cwd=REPO_DIR)


[Ejecutando] C:\Users\VICTORIA\anaconda3\envs\imagen\python.exe -m pip install --upgrade pip setuptools wheel


[Ejecutando] C:\Users\VICTORIA\anaconda3\envs\imagen\python.exe -m pip install matplotlib opencv-python pillow numpy ipywidgets requests


[Ejecutando] C:\Users\VICTORIA\anaconda3\envs\imagen\python.exe -m pip install basicsr facexlib


[Ejecutando] C:\Users\VICTORIA\anaconda3\envs\imagen\python.exe -m pip install -r requirements.txt


[Ejecutando] C:\Users\VICTORIA\anaconda3\envs\imagen\python.exe -m pip install realesrgan


[Ejecutando] C:\Users\VICTORIA\anaconda3\envs\imagen\python.exe setup.py develop

C:\Users\VICTORIA\anaconda3\envs\imagen\lib\site-packages\setuptools\__init__.py:92: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip insta

(0,
 "C:\\Users\\VICTORIA\\anaconda3\\envs\\imagen\\lib\\site-packages\\setuptools\\__init__.py:92: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.\n!!\n\n        ********************************************************************************\n        Requirements should be satisfied by a PEP 517 installer.\n        If you are using pip, you can try `pip install --use-pep517`.\n\n        This deprecation is overdue, please update your project and remove deprecated\n        calls to avoid build errors in the future.\n        ********************************************************************************\n\n!!\n  dist.fetch_build_eggs(dist.setup_requires)\nC:\\Users\\VICTORIA\\anaconda3\\envs\\imagen\\lib\\site-packages\\setuptools\\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.\n!!\n\n        ********************************************************************************\n        Please consider removing the following cl

## 7. Verificación posterior a la instalación

Prueba la importación de los módulos clave.

In [9]:
mods = ["torch", "cv2", "basicsr", "facexlib", "gfpgan"]
for m in mods:
    try:
        __import__(m)
        print(f"{m}: OK")
    except Exception as e:
        print(f"{m}: ERROR -> {e}")

torch: OK
cv2: OK
basicsr: ERROR -> No module named 'torchvision.transforms.functional_tensor'
facexlib: OK
gfpgan: ERROR -> No module named 'gfpgan'


## 8. Descarga de pesos preentrenados

La documentación oficial de GFPGAN muestra la descarga del modelo v1.3 hacia `experiments/pretrained_models`. El script de inferencia también soporta `1.4` y `RestoreFormer`; si el peso no existe, el propio script puede usar la URL remota, pero descargarlo antes suele ser más estable en local. citeturn391527view0turn859644view0

In [10]:
import requests

MODEL_URLS = {
    "1": "https://github.com/TencentARC/GFPGAN/releases/download/v0.1.0/GFPGANv1.pth",
    "1.2": "https://github.com/TencentARC/GFPGAN/releases/download/v0.2.0/GFPGANCleanv1-NoCE-C2.pth",
    "1.3": "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth",
    "1.4": "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
    "RestoreFormer": "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/RestoreFormer.pth",
}

MODEL_FILENAMES = {
    "1": "GFPGANv1.pth",
    "1.2": "GFPGANCleanv1-NoCE-C2.pth",
    "1.3": "GFPGANv1.3.pth",
    "1.4": "GFPGANv1.4.pth",
    "RestoreFormer": "RestoreFormer.pth",
}

SELECTED_MODEL = "1.4"   # cambia a "1.3", "1.2", "1" o "RestoreFormer" si quieres

def download_file(url, dst, chunk_size=1024*1024):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        print(f"Ya existe: {dst.name}")
        return dst
    print(f"Descargando {url}\n-> {dst}")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(dst, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    print("Descarga completa:", dst)
    return dst

weight_path = WEIGHTS_DIR_1 / MODEL_FILENAMES[SELECTED_MODEL]
download_file(MODEL_URLS[SELECTED_MODEL], weight_path)
print("Peso disponible en:", weight_path)

Descargando https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth
-> C:\Users\VICTORIA\GFPGAN\experiments\pretrained_models\GFPGANv1.4.pth
Descarga completa: C:\Users\VICTORIA\GFPGAN\experiments\pretrained_models\GFPGANv1.4.pth
Peso disponible en: C:\Users\VICTORIA\GFPGAN\experiments\pretrained_models\GFPGANv1.4.pth


## 9. Carpeta de entrada

Coloca aquí tus imágenes:

In [12]:
from pathlib import Path

INPUT_DIR = Path(r"C:\Users\VICTORIA\Documents\Python Scripts\GFPGAN\inputs\user_images")
OUTPUT_DIR = Path(r"C:\Users\VICTORIA\Documents\Python Scripts\GFPGAN\results\user_results")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de entrada:", INPUT_DIR)
print("Carpeta de salida:", OUTPUT_DIR)

Carpeta de entrada: C:\Users\VICTORIA\Documents\Python Scripts\GFPGAN\inputs\user_images
Carpeta de salida: C:\Users\VICTORIA\Documents\Python Scripts\GFPGAN\results\user_results


In [13]:
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Coloca aquí tus imágenes:")
print(INPUT_DIR)

Coloca aquí tus imágenes:
C:\Users\VICTORIA\Documents\Python Scripts\GFPGAN\inputs\user_images


## 10. Listar imágenes de entrada

In [19]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
image_files = sorted([p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

print(f"Imágenes encontradas: {len(image_files)}")
for p in image_files:
    print("-", p.name)

Imágenes encontradas: 0


## 11. Visualización de entradas

In [ ]:
import cv2
import math
import matplotlib.pyplot as plt

def imread_rgb(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {img_path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def show_grid(paths, cols=3, title="Imágenes"):
    if not paths:
        print("No hay imágenes para mostrar.")
        return
    n = len(paths)
    rows = math.ceil(n / cols)
    plt.figure(figsize=(5 * cols, 4 * rows))
    for i, path in enumerate(paths, start=1):
        plt.subplot(rows, cols, i)
        plt.imshow(imread_rgb(path))
        plt.title(path.name, fontsize=10)
        plt.axis("off")
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

show_grid(image_files[:12], cols=3, title="Entradas")

## 12. Configuración de inferencia

El script oficial acepta parámetros como:
- `-i` entrada
- `-o` salida
- `-v` versión del modelo
- `-s` upscale
- `--bg_upsampler realesrgan`
- `--only_center_face`
- `--aligned`
- `--ext`

En CPU, el propio script desactiva Real-ESRGAN para el fondo porque esa parte es lenta sin CUDA. citeturn859644view0

In [ ]:
# Configuración editable
MODEL_VERSION = SELECTED_MODEL        # "1", "1.2", "1.3", "1.4", "RestoreFormer"
UPSCALE = 2
ONLY_CENTER_FACE = False
ALIGNED = False
EXT = "auto"                         # auto | jpg | png
SUFFIX = "gfpgan_local"
WEIGHT_BLEND = 0.7                   # parámetro -w del script

try:
    import torch
    USE_BG_UPSAMPLER = torch.cuda.is_available()
except Exception:
    USE_BG_UPSAMPLER = False

BACKGROUND_UPSAMPLER = "realesrgan" if USE_BG_UPSAMPLER else "none"

print("MODEL_VERSION =", MODEL_VERSION)
print("UPSCALE =", UPSCALE)
print("BACKGROUND_UPSAMPLER =", BACKGROUND_UPSAMPLER)
print("ONLY_CENTER_FACE =", ONLY_CENTER_FACE)
print("ALIGNED =", ALIGNED)
print("EXT =", EXT)
print("SUFFIX =", SUFFIX)
print("WEIGHT_BLEND =", WEIGHT_BLEND)

## 13. Ejecutar GFPGAN sobre toda la carpeta

Si no hay imágenes, la celda no corre.

In [ ]:
if not image_files:
    print("No hay imágenes en INPUT_DIR. Copia primero tus archivos.")
else:
    cmd = [
        sys.executable,
        "inference_gfpgan.py",
        "-i", str(INPUT_DIR),
        "-o", str(OUTPUT_DIR),
        "-v", str(MODEL_VERSION),
        "-s", str(UPSCALE),
        "-w", str(WEIGHT_BLEND),
        "--ext", str(EXT),
        "--suffix", str(SUFFIX),
    ]

    if BACKGROUND_UPSAMPLER and BACKGROUND_UPSAMPLER.lower() != "none":
        cmd += ["--bg_upsampler", BACKGROUND_UPSAMPLER]

    if ONLY_CENTER_FACE:
        cmd += ["--only_center_face"]

    if ALIGNED:
        cmd += ["--aligned"]

    run_command(cmd, cwd=REPO_DIR)

## 14. Mostrar archivos de salida

In [ ]:
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))

## 15. Cargar resultados restaurados

In [ ]:
possible_dirs = [
    OUTPUT_DIR / "restored_imgs",
    OUTPUT_DIR / "restored_faces",
    OUTPUT_DIR / "cmp",
    OUTPUT_DIR / "cropped_faces",
    OUTPUT_DIR,
]

result_images = []
for d in possible_dirs:
    if d.exists():
        result_images.extend([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

# quitar duplicados conservando orden
seen = set()
dedup = []
for p in result_images:
    if p not in seen:
        seen.add(p)
        dedup.append(p)

result_images = dedup
print(f"Resultados encontrados: {len(result_images)}")
for p in result_images[:20]:
    print("-", p)

## 16. Visualizar resultados

In [ ]:
show_grid(result_images[:12], cols=3, title="Resultados GFPGAN")

## 17. Emparejar entrada y salida automáticamente

In [ ]:
from collections import defaultdict

inputs_by_stem = {p.stem: p for p in image_files}
results_by_stem = defaultdict(list)

for p in result_images:
    results_by_stem[p.stem].append(p)

pairs = []
for stem, inp in inputs_by_stem.items():
    candidates = []
    for rstem, rpaths in results_by_stem.items():
        if stem in rstem:
            candidates.extend(rpaths)
    # priorizar restored_imgs si existe
    candidates = sorted(candidates, key=lambda x: ("restored_imgs" not in str(x), len(str(x))))
    if candidates:
        pairs.append((inp, candidates[0]))

print(f"Pares encontrados: {len(pairs)}")
for inp, out in pairs[:10]:
    print(inp.name, "->", out.name)

## 18. Comparación antes / después

In [ ]:
if not pairs:
    print("No se pudieron emparejar automáticamente entradas y salidas.")
else:
    n = len(pairs)
    plt.figure(figsize=(12, 4 * n))
    for i, (inp, out) in enumerate(pairs, start=1):
        plt.subplot(n, 2, 2*i - 1)
        plt.imshow(imread_rgb(inp))
        plt.title(f"Entrada: {inp.name}")
        plt.axis("off")

        plt.subplot(n, 2, 2*i)
        plt.imshow(imread_rgb(out))
        plt.title(f"Salida: {out.name}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 19. Proceso de una sola imagen

Útil para depuración.

In [ ]:
SINGLE_IMAGE_NAME = image_files[0].name if image_files else None
print("Imagen seleccionada:", SINGLE_IMAGE_NAME)

In [ ]:
if SINGLE_IMAGE_NAME is None:
    print("No hay imágenes en INPUT_DIR.")
else:
    single_input = INPUT_DIR / SINGLE_IMAGE_NAME
    single_output = OUTPUT_DIR / "single_run"
    single_output.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "inference_gfpgan.py",
        "-i", str(single_input),
        "-o", str(single_output),
        "-v", str(MODEL_VERSION),
        "-s", str(UPSCALE),
        "-w", str(WEIGHT_BLEND),
        "--ext", str(EXT),
        "--suffix", str(SUFFIX),
    ]

    if BACKGROUND_UPSAMPLER and BACKGROUND_UPSAMPLER.lower() != "none":
        cmd += ["--bg_upsampler", BACKGROUND_UPSAMPLER]

    if ONLY_CENTER_FACE:
        cmd += ["--only_center_face"]

    if ALIGNED:
        cmd += ["--aligned"]

    run_command(cmd, cwd=REPO_DIR)
    print("Proceso terminado. Carpeta de salida:", single_output)

## 20. Prueba directa del kernel y de GPU

Esto confirma que el notebook usa el Python correcto.

In [ ]:
import sys
print("Kernel Python:", sys.executable)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        x = torch.rand(2, 2).cuda()
        print("Tensor en GPU:", x)
except Exception as e:
    print("Prueba GPU:", e)

## 21. Solución de errores frecuentes

### A. `Torch ... +cpu`
Instalaste la versión CPU de PyTorch. Reinstala una build con CUDA.

### B. `CUDA disponible: False`
Posibles causas:
- Jupyter está usando otro entorno
- el driver NVIDIA no está activo
- instalaste una rueda CPU de PyTorch

### C. `ModuleNotFoundError: gfpgan`
Ejecuta de nuevo la celda de instalación y confirma que el kernel apunta al entorno correcto.

### D. `git` no reconocido
Instala Git para Windows y vuelve a ejecutar la celda del clon.

### E. El modelo no descarga
Revisa conexión a Internet o descarga manualmente el `.pth` y cópialo a:
- `experiments/pretrained_models`
- o `gfpgan/weights`

### F. Fondo muy lento en CPU
Usa `BACKGROUND_UPSAMPLER = "none"`.

### G. Si Jupyter usa otro entorno
Dentro del notebook ejecuta:
```python
import sys
print(sys.executable)
```
Debe apuntar a tu entorno de Anaconda correcto.